# Custom CORENN + Hybrid Correlator demo

Demonstrates the standalone modules that accompany `pct.py`:

- **`corenn.py`** — a transparent, tested re-implementation of the CORENN PSD inversion (non-linear weighted least squares on the un-linearized Siegert model, cubic B-spline / P-spline discretization, Tikhonov curvature penalty, GCV regularization, non-negativity, *free baseline*), plus improvements: replicate-calibrated Schätzel noise weighting, uncertainty bands, lag-window control, and a multi-angle form-factor inversion.
- **`hybrid_correlator.py`** — extends the usable lag range by software-correlating the photon count-rate trace across replicates and *gracefully merging* it with the hardware correlogram via inverse-variance weighting.

Theory: `docs/theory/extended_fitting_theory.tex`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import corenn as C
import hybrid_correlator as H

# Point this at one LS Instruments measurement directory (contains Repetition */):
MEAS = ('/Users/mwhittaker/Library/CloudStorage/GoogleDrive-mwhittaker@lbl.gov/'
        'My Drive/PCT/Data/LiTFSI/20m/22C/Measurement 1')

## 1. Ground-truth validation on a synthetic bimodal sample
CORENN's signature capability is resolving a bimodal PSD. We build a 20 nm + 200 nm mixture, add noise, and invert with bootstrap uncertainty bands.

In [ ]:
rng = np.random.default_rng(0)
T, eta, n_sol = 295.0, 0.94e-3, 1.33
q = C.q_from_angle(90.0, n_sol, 632.8)
tau = np.geomspace(1.6e-6, 10.0, 200)
R_true = np.array([20e-9, 200e-9]); wfrac = np.array([0.5, 0.5])
D = C.diffusion_coefficient(R_true, T, eta)
beta = 0.34
reps = [beta*sum(wi*np.exp(-q*q*d*tau) for wi,d in zip(wfrac,D))**2
        + 2e-3*rng.standard_normal(tau.size) for _ in range(5)]
ds = C.DLSDataset(tau=tau, g2m=np.mean(reps,0), q=q, beta=beta,
                  temperature=T, viscosity=eta, n_solvent=n_sol)
res = C.corenn_invert(ds, radius_range=(2e-9,1e-6), n_basis=22, n_R=140,
                      replicates=reps, uncertainty='bootstrap', n_bootstrap=80)
print(res.summary())

fig, ax = plt.subplots(figsize=(6,4))
ax.plot(res.R*1e9, res.R*res.intensity, lw=2, label='recovered')
if res.intensity_band is not None:
    ax.fill_between(res.R*1e9, res.R*res.intensity_band[0], res.R*res.intensity_band[1], alpha=.25)
for r in R_true*1e9: ax.axvline(r, color='k', ls='--', alpha=.5)
ax.set_xscale('log'); ax.set_xlabel('radius [nm]'); ax.set_ylabel('R·I(R)')
ax.set_title('Synthetic bimodal recovery'); ax.legend(); plt.show()

## 2. Hardware-only CORENN on real data
Load all replicates of one measurement (used for empirical noise calibration) and invert the hardware correlogram.

In [ ]:
base, hw_reps = C.load_measurement_replicates(MEAS)
print(f'q={base.q:.3e} 1/m  beta={base.beta:.3f}  T={base.temperature} K  '
      f'eta={base.viscosity} Pa.s  duration={base.duration} s')
res_hw = C.corenn_invert(base, radius_range=(5e-10,5e-5), n_basis=30, n_R=200,
                         replicates=hw_reps, fit_baseline=True)
print(res_hw.summary())

## 3. Hybrid correlator: software-correlate the count traces and merge
The hardware correlogram is trustworthy through the main decay; the long-lag dynamics are better recovered from a software cross-correlation of the count-rate traces, ensemble-averaged over replicates. The two are blended by inverse-variance weighting.

In [ ]:
ds_hy, merged = H.build_hybrid_dataset(MEAS)
hw = merged.components['hardware']; sw = merged.components['software']; Wd = merged.components['weights']
frac_sw = Wd['w_sw']/(Wd['w_hw']+Wd['w_sw'])
cross = Wd['tau'][np.argmin(np.abs(frac_sw-0.5))]
print(f'software->hardware scale={merged.scale_sw:.3f}; crossover ~ {cross:.3f} s')

fig, (a1,a2) = plt.subplots(1,2, figsize=(12,4))
a1.plot(hw['tau'], hw['g2m'], '.', ms=3, alpha=.5, label='hardware')
a1.plot(sw['tau'], sw['g2m'], '.', ms=6, alpha=.7, label='software')
a1.plot(ds_hy.tau, ds_hy.g2m, 'k-', lw=1.2, label='merged')
a1.set_xscale('log'); a1.set_xlabel('tau [s]'); a1.set_ylabel('g2-1'); a1.legend(); a1.set_title('Merge')
a2.plot(Wd['tau'], 1-frac_sw, label='hardware'); a2.plot(Wd['tau'], frac_sw, label='software')
a2.axvline(cross, color='k', ls=':'); a2.set_xscale('log'); a2.set_xlabel('tau [s]')
a2.set_ylabel('blend weight'); a2.legend(); a2.set_title('Inverse-variance weights'); plt.show()

## 4. CORENN on the hybrid curve vs hardware-only
The extended lag range lets the inversion separate the main mode from a slow aggregate tail.

In [ ]:
res_hy = C.corenn_invert(ds_hy, radius_range=(5e-10,5e-5), n_basis=30, n_R=200, fit_baseline=True)
print(res_hy.summary())

fig, ax = plt.subplots(figsize=(6,4))
ax.plot(res_hw.R*1e9, res_hw.R*res_hw.intensity, lw=2, label='hardware only')
ax.plot(res_hy.R*1e9, res_hy.R*res_hy.intensity, lw=2, label='hybrid')
ax.axvline(197.9, color='k', ls='--', alpha=.5, label='vendor CORENN ~198 nm')
ax.set_xscale('log'); ax.set_xlabel('radius [nm]'); ax.set_ylabel('R·I(R)')
ax.legend(); ax.set_title('Recovered PSD'); plt.show()

## 5. Multi-angle joint inversion (form-factor weighting)
When several angles are available, `corenn_multi_angle` solves for a single q-consistent distribution, folding the angular dependence through the sphere form factor P(qR). Pass a list of `DLSDataset` at different q (e.g. from `C.build_dataset_from_repetition` over angle-resolved measurements).

In [ ]:
# Example skeleton (fill in angle-resolved measurement directories):
# datasets = [C.build_dataset_from_repetition(d) for d in angle_rep_dirs]
# res_ma = C.corenn_multi_angle(datasets, radius_range=(5e-9,1e-6), use_form_factor=True)
# print([f"{p['radius_nm']:.0f} nm ({p['rel_amount']*100:.0f}%)" for p in res_ma.peaks])

In [1]:
import numpy as np

In [6]:
np.logspace(np.log10(10), np.log10(15), 10)

array([10.        , 10.46081919, 10.94287381, 11.44714243, 11.97464871,
       12.5264635 , 13.10370697, 13.70755093, 14.33922118, 15.        ])

In [5]:
np.logspace(np.log10(16), np.log10(100), 30)

array([ 16.        ,  17.04370942,  18.15550192,  19.33981869,
        20.60139062,  21.94525721,  23.37678668,  24.90169746,
        26.52608097,  28.256426  ,  30.09964462,  32.06309977,
        34.15463472,  36.38260434,  38.75590851,  41.28402768,
        43.97706071,  46.84576524,  49.90160063,  53.15677379,
        56.62428788,  60.31799428,  64.25264795,  68.44396631,
        72.90869207,  77.6646601 ,  82.73086867,  88.12755534,
        93.8762778 , 100.        ])